# Session 4 — Modeling

Five-model Telecom Customer Churn modeling workflow.

**Order:** setup → load Session 3 outputs → load preprocessor → define 5 models → wrap each in a pipeline → cross-validation ROC-AUC → fit/predict → metrics table → classification reports → select winner by mean CV AUC → save all fitted pipelines.

In [ ]:
import os
import warnings
import joblib
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

In [ ]:
X_train = pd.read_csv("../data/clean_X_train.csv")
X_test = pd.read_csv("../data/clean_X_test.csv")
y_train = pd.read_csv("../data/y_train.csv").squeeze("columns")
y_test = pd.read_csv("../data/y_test.csv").squeeze("columns")

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

In [ ]:
preprocessor = joblib.load("../src/preprocessor.joblib")
print("Preprocessor loaded successfully:", type(preprocessor).__name__)

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
    "Extra Trees": ExtraTreesClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
    "KNN": KNeighborsClassifier()
}

print(list(models.keys()))

In [ ]:
pipelines = {}

for name, model in models.items():
    pipelines[name] = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

print("Five pipelines created.")

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []

for name, pipe in pipelines.items():
    scores = cross_val_score(
        pipe, X_train, y_train,
        cv=cv, scoring="roc_auc", n_jobs=-1
    )
    cv_rows.append({
        "Model": name,
        "CV AUC Mean": scores.mean(),
        "CV AUC Std": scores.std(),
        "CV AUC Min": scores.min(),
        "CV AUC Max": scores.max()
    })

cv_df = pd.DataFrame(cv_rows).sort_values("CV AUC Mean", ascending=False).reset_index(drop=True)
cv_df

In [ ]:
test_rows = []
reports = {}

for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]

    test_rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_test, y_prob)
    })

    reports[name] = classification_report(
        y_test, y_pred,
        target_names=["No Churn", "Churn"],
        zero_division=0
    )

metrics_df = pd.DataFrame(test_rows).sort_values("ROC_AUC", ascending=False).reset_index(drop=True)
metrics_df

In [ ]:
comparison_df = cv_df.merge(metrics_df, on="Model", how="left")
comparison_df

In [ ]:
for model_name, report in reports.items():
    print("=" * 70)
    print(model_name)
    print("=" * 70)
    print(report)

In [ ]:
winner = cv_df.iloc[0]["Model"]
winner_cv_auc = cv_df.iloc[0]["CV AUC Mean"]

print("Selected model for Session 5:", winner)
print(f"Mean CV AUC: {winner_cv_auc:.4f}")

In [ ]:
os.makedirs("../models/session4", exist_ok=True)
os.makedirs("../outputs", exist_ok=True)

for name, pipe in pipelines.items():
    filename = name.lower().replace(" ", "_") + ".pkl"
    joblib.dump(pipe, f"../models/session4/{filename}")

comparison_df.to_csv("../outputs/model_comparison.csv", index=False)
cv_df.to_csv("../outputs/cross_validation_auc.csv", index=False)

print("Session 4 artifacts saved.")

In [ ]:
for name in pipelines:
    filename = name.lower().replace(" ", "_") + ".pkl"
    loaded = joblib.load(f"../models/session4/{filename}")
    print(f"{name}: reload OK")